Import library

In [34]:
import os
import sys
from tqdm import tqdm
import glob
import typing
import import_ipynb

# Add current directory to path for imports
import os
current_dir = "/home2/ducvu/speech-processing-implement/codes/feature_extraction"
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

import importlib
import config_classifiers
from config_classifiers import *
config_classifiers = importlib.reload(config_classifiers)

import numpy as np
import pandas as pd
from collections import defaultdict
import librosa, opensmile

from sklearn.preprocessing import binarize,scale, robust_scale
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.svm import SVR, SVC, LinearSVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score, precision_score, recall_score, roc_auc_score, r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor

from sklearn.metrics import precision_recall_fscore_support as score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer


In [35]:
def read_file(file_path:str) -> pd:
    f = open(file_path, 'r')
    return f.read()

In [36]:
def calcualte_metrics(actual_labels:list, pred_vals:list, avg=None) -> tuple:
    if avg == None:
        f1_val, pres_val, rec_val, conf_val = score(actual_labels, pred_vals, average=None)
    else:
        f1_val, pres_val, rec_val, conf_val = score(actual_labels, pred_vals, average=avg)
    return f1_val, pres_val, rec_val, conf_val

In [37]:
def majority_voting_labels(df:pd, a_class_type:list, verbose:bool) -> tuple[int]:
    data = {
        "r_IDs": df[["r_IDs", "pred_label"]].groupby("r_IDs").mean().index.values,
        "grouped_pred_label": df[["r_IDs", "pred_label"]].groupby("r_IDs").mean().pred_label.values
    }
    df_final_test_grouped = df.merge(pd.DataFrame(data), on="r_IDs", how="inner")
    print(f"df_final_test_grouped: {df_final_test_grouped}")

    df_final_test_grouped = df_final_test_grouped.drop_duplicates(subset=["r_IDs"], keep="first")
    print(f"df_final_test_grouped_without_duplicates: {df_final_test_grouped}")

    if a_class_type == "3-way":
        threshold_val = 0.3333333
        final_pred_label_list = []
        for x in df_final_test_grouped.grouped_pred_label:
            if x < threshold_val:
                final_pred_label_list.append(0)
            elif x >= threshold_val and x < (2 * threshold_val):
                final_pred_label_list.append(1)
            else:
                final_pred_label_list.append(2)
        df_final_test_grouped["pred_label"] = final_pred_label_list
    else:
        threshold_val = 0.5
        final_pred_label_list = []
        for x in df_final_test_grouped.grouped_pred_label:
            if x < threshold_val:
                final_pred_label_list.append(0)
            else:
                final_pred_label_list.append(1)
    df_final_test_grouped.insert(len(df_final_test_grouped.columns), "final_pred_label", final_pred_label_list)


    actual_labels = df_final_test_grouped.labels

    # Calculate the initial results
    col_2_cons = "final_pred_label"
    pred_vals = df_final_test_grouped["col_2_cons"]
    f1_val, pres_val, rec_val, conf_val = calcualte_metrics(actual_labels, pred_vals, avg="macro")
    
    if verbose == 1:
        print(f"Macro F1-score: {round(f1_val, 2)}")
        print(f"Macro Precision: {round([pres_val, 2])}")
        print(f"Macro Recall: {round(rec_val, 2)}")
        print(f"Conf matrix: {conf_val}")
    
    if a_class_type == "3-way":
        precision, recall, fscore, support = score(actual_labels, pred_vals)
        if verbose == 1:
            print("Metric \t HC \t MCI \t Dementia")
            print(f"Precision \t {round(precision[0], 2)} \t {round(precision[1], 2)}, \t {round(precision[2], 2)}")
            print(f"Precision \t {round(recall[0], 2)} \t {round(recall[1], 2)}, \t {round(recall[2], 2)}")
            print(f"Precision \t {round(fscore[0], 2)} \t {round(fscore[1], 2)}, \t {round(fscore[2], 2)}")

    return f1_val, pres_val, rec_val, conf_val

In [38]:
def analyze_metadata(df_metadata_final:pd) -> dict:

    # Dict for saving result
    stat_result = defaultdict()

    diagnosis_group = df_metadata_final.diagnosis.unique()

    # Diagnosis distribution
    diagnosis_count = df_metadata_final.diagnosis.value_counts()
    print(f"All data: {len(df_metadata_final)} and diagnosis count: {diagnosis_count}")

    # Age distribution
    age_stat = defaultdict()
    for diagnosis in diagnosis_group:
        group_data = df_metadata_final[df_metadata_final.diagnosis == diagnosis].age
        mean_age = np.mean(group_data)
        std_age = np.std(group_data)

        age_stat[diagnosis] = {
            "Mean age": mean_age,
            "Std age": std_age,
        }

    age_stat["Overall"] = {
        "Mean age": np.mean(df_metadata_final.age),
        "Std age": np.std(df_metadata_final.age),
    }

    bin_range = (0, 50, 60, 70, 80, 100)
    bin_labels = ["0 - 50", "50 - 60", "60 - 70", "70 - 80", "80 - 100"]

    age_histograms = defaultdict()
    for diagnosis in diagnosis_group:
        group_data = df_metadata_final[df_metadata_final.diagnosis == diagnosis].age
        hist, _ = np.histogram(group_data, bins=bin_range)

        age_histograms[diagnosis] = dict(zip(bin_labels, hist))

    stat_result["age stat"] = age_stat
    stat_result["age histograms"] = age_histograms

    # Ethnicity Distribution in disease
    ethnic_diagnosis = defaultdict()
    ethnic_group = df_metadata_final.ethnicity.unique()
    for ethnic in ethnic_group:
        diagnosis_count = df_metadata_final[df_metadata_final.ethnicity == ethnic].diagnosis.value_counts()
        ethnic_diagnosis[ethnic] = diagnosis_count

    stat_result["Ethnic diagnosis"] = ethnic_diagnosis

    # Gender distribution
    gender_diagnosis = defaultdict()
    for diagnosis in diagnosis_group:
        gender_count = df_metadata_final[df_metadata_final.diagnosis == diagnosis].age.value_counts()
        gender_diagnosis[diagnosis] = gender_count

    stat_result["gender diagnosis"] = gender_diagnosis


    return stat_result

In [39]:
def extract_acoustic_features(df_metadata_final, 
                                list_task_name:list[str], 
                                data_path:str, 
                                feat_path:str, 
                                list_feature_type:list[str], 
                                force_reextract=False) -> defaultdict:

    LIST_TASKS = config_classifiers.LIST_TASKS

    os.makedirs(feat_path, exist_ok=True)

    feat_dict = defaultdict()

    for openSmile in list_feature_type:
        print(f"Processing feature type: {openSmile}")
        print(50*"-")
        df_feat_name = feat_path + "cognoSpeak_" + openSmile + ".csv"

        # Check feature exist
        if os.path.exists(df_feat_name) and not force_reextract:
            print(f"Loading already exist: {df_feat_name}")
            df_feat = pd.read_csv(df_feat_name)

            if openSmile == "eGeMAPSv02":
                smile = opensmile.Smile(
                    feature_set=opensmile.FeatureSet.eGeMAPSv02,
                    feature_level = opensmile.FeatureLevel.Functionals,
                )
            elif openSmile == "ComParE_2016":
                smile = opensmile.Smile(
                    feature_set=opensmile.FeatureSet.ComParE_2016,
                    feature_level = opensmile.FeatureLevel.Functionals,
                )

            feat_names = list(smile.feature_names)
            print(f"Feature names: {feat_names}")
            feat_dict[openSmile] = (df_feat, feat_names)
            continue
            
        # If file is not exist
        if openSmile == "eGeMAPSv02":
            smile = opensmile.Smile(
                feature_set=opensmile.FeatureSet.eGeMAPSv02,
                feature_level = opensmile.FeatureLevel.Functionals,
            )
        elif openSmile == "ComParE_2016":
            smile = opensmile.Smile(
                feature_set=opensmile.FeatureSet.ComParE_2016,
                feature_level = opensmile.FeatureLevel.Functionals,
            )
        else:
            print(f"Unknow feature: {openSmile}")

        # create data frame for containing feature
        df_feat = pd.DataFrame([], columns=["participant_id", "task"] + smile.feature_names)

        # Remove index columns
        df_metadata_reset = df_metadata_final.reset_index(drop=True)

        extract_error = []
        for ind in tqdm(df_metadata_reset.index, desc=f"Extracting {openSmile}"):
            for task in list_task_name:
                audio_pattern = f"{data_path}/audio_files/{df_metadata_reset["participant_id"][ind]}/{df_metadata_reset["participant_id"][ind]}_{LIST_TASKS[task]["file_pattern"]}"
                print(f"audio pattern: {audio_pattern}")
                audio_file = glob.glob(audio_pattern)

                # Check audio exist
                if len(audio_file) == 0:
                    extract_error.append(
                        {
                            "subject": df_metadata_reset["participant_id"][ind],
                            "question": task,
                            "error": "Audio file not found"
                        }
                    )
                    continue

                an_audio = audio_file[0]
                try:
                    # Load audio and extract feature
                    sig, sample_rate = librosa.load(an_audio, sr=None)
                    df_os_feat = smile.process_signal(sig, sample_rate)
                    df_os_feat = df_os_feat.reset_index(drop=True)

                    # Add to metadata
                    df_os_feat.insert(0, "participant_id", df_metadata_reset["participant_id"][ind])
                    df_os_feat.insert(1, "task", task)

                    # Concate to main dataframe
                    df_feat = pd.concat([df_feat, df_os_feat], ignore_index=True, sort=False)

                except Exception as e:
                    extract_error.append(
                        {
                            "subject": df_metadata_reset["participant_id"][ind],
                            "question": task,
                            "error": str(e)
                        }
                    )

                
        if extract_error:
            print(f"\n  WARNING: {len(extract_error)} extraction errors occurred:")
            for err in extract_error[:5]:  # Show first 5 errors
                print(f"    - {err['subject']}, {err['question']}: {err['error']}")
            if len(extract_error) > 5:
                print(f"    ... and {len(extract_error) - 5} more errors")

        
        print(f"Merge features with metadata")
        df_feat = df_feat.merge(df_metadata_final, on="participant_id", how="inner")

        # Save feature
        print(f"Saving feature to: {df_feat_name}")
        df_feat.to_csv(df_feat_name, index=False)

        # Extract feature name
        feat_names = list(smile.feature_names)
        print(f"Successfully extract {len(feat_names)} feature from {len(df_feat)} samples")

        feat_dict[openSmile] = (df_feat, feat_names)

    print(50*"-")
    print("Complete feature extraction")
    for feat_type, (df, names) in feat_dict.items():
        print(f"{feat_type}: {len(df)} samples, {len(names)} features")

    return feat_dict


In [40]:
def kfold_split(df_metadata_final:pd, 
                n_folds:int,
                label_map:dict,
                label_column:str="diagnosis",
                subject_id_column:str="participant_id",
                random_state:int=42) -> pd:

    df_metadata_copy = df_metadata_final.copy()
    # Find unique participants and their labels
    participants = df_metadata_copy[subject_id_column].unique()

    pariticpant_labels = []
    for p in participants:
        # Taking first value if duplicate
        label = df_metadata_copy[df_metadata_copy[subject_id_column] == p][label_column].iloc[0]
        pariticpant_labels.append(label)


    # Convert diagnosis to numeric
    if isinstance(pariticpant_labels[0], str):
        numeric_labels = [label_map[l] for l in pariticpant_labels]
    else:
        numeric_labels = pariticpant_labels

    df_metadata_copy["label"] = df_metadata_copy[label_column].map(label_map)
    

    # Create stratified k-fold splits
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    print(skf)

    # Add FOLD columns
    for k in range(n_folds):
        df_metadata_copy[f"FOLD_{k}"] = ''
    
    print(skf.split(participants, numeric_labels))
    # Assign train/test label for each fold
    for fold_idx ,(train_idx, test_idx) in enumerate(skf.split(participants, numeric_labels)):
        train_participants = participants[train_idx]
        test_participants = participants[test_idx]

        for p in train_participants:
            df_metadata_copy.loc[df_metadata_copy[subject_id_column] == p, f"FOLD_{fold_idx}"] = "TRAIN"
        for p in test_participants:
            df_metadata_copy.loc[df_metadata_copy[subject_id_column] == p, f"FOLD_{fold_idx}"] = "TEST"
    return df_metadata_copy

In [41]:
def train_classifier(df_train:pd,
                    feat_names:list[str],
                    classifier_type:str,
                    class_type_chosen:str,
                    n_jobs:int):
    # Prepare data
    x_train = np.array(df_train[feat_names])
    y_train = np.array(df_train["label"].values)

    # Scale feature
    x_train_scaled = robust_scale(x_train)
    if class_type_chosen == "grid":
        pass
    elif class_type_chosen == "simple":
       # Use default parameters
        if classifier_type == 'LR':
            model = LogisticRegression(max_iter=int(1e+20), n_jobs=n_jobs)
        elif classifier_type == 'SVM':
            model = SVC(probability=True)
        elif classifier_type == 'MLP':
            model = MLPClassifier(random_state=1, max_iter=int(1e+20))
        else:
            raise ValueError(f"Unknown classifier type: {classifier_type}")
        
        trained_model = model.fit(x_train_scaled, y_train)
    
    else:
        raise ValueError(f"Unknown optimization type: {class_type_chosen}")
    return trained_model 
    

def evaluate_fold(trained_model, 
                    df_test:pd, 
                    feat_names:list[str], 
                    way_classification:list[str], 
                    majority_voting_labels:tuple) -> tuple[int]:
    pass

def run_single_experiment(classifier_chosen:list[int],
                            task_chosen:list[str],
                            class_type_chosen:list[str],
                            opensmile_feature:list[str],
                            N_FOLDS:int,
                            CV_SCORER:str,
                            N_JOBS:int,
                            majority_voting_labels:tuple) -> defaultdict:
    pass

In [42]:
# Hyperparameter
CV_SCORER = config_classifiers.CV_SCORER
N_FOLDS = config_classifiers.N_FOLDS

# Directory
DATA_PATH = config_classifiers.DATA_PATH
FEATS_PATH = config_classifiers.FEATS_PATH
RESULTS_PATH = config_classifiers.RESULTS_PATH

# List acoustic feature
LIST_ACOUSTIC = config_classifiers.LIST_ACOUSTIC

# List classifier and chosen
LIST_CLASSIFIER_NAME = config_classifiers.LIST_CLASSIFIER_NAME
CLASSIFIER_CHOSEN = config_classifiers.CLASSIFIER_CHOSEN

# List task chosen
TASK_CHOSEN = config_classifiers.TASK_CHOSEN

# List class type chosen
CLASS_TYPE_CHOSEN = config_classifiers.CLASS_TYPE_CHOSEN

# Way for classifying
WAY_CLASSIFICATION = config_classifiers.WAY_CLASSIFICATION

# Mapping label
LABEL_MAP = config_classifiers.LABEL_MAP

In [43]:
df_metadata_final = pd.read_csv(f"{DATA_PATH}/metadata.csv")
for i in range(N_FOLDS):
    df_metadata_final = df_metadata_final.drop(columns=f"FOLD_{i}")

df_metadata_final = df_metadata_final.drop(columns=f"labels")
df_metadata_final

,participant_id,age,gender,diagnosis,ethnicity
0,participant_001,62,F,HC,White-British
1,participant_002,58,M,HC,White-British
2,participant_003,69,F,HC,Asian
3,participant_004,64,M,HC,Other
4,participant_005,71,F,HC,White-British
5,participant_006,78,M,Dementia,White-British
6,participant_007,82,F,Dementia,White-British
7,participant_008,75,M,Dementia,Asian
8,participant_009,80,F,Dementia,Mixed
9,participant_010,77,M,Dementia,White-British


In [44]:
df_metadata_final = kfold_split(df_metadata_final, 5, LABEL_MAP, "diagnosis", "participant_id", 42)
df_metadata_final

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
<generator object _BaseKFold.split at 0x7f6b035b9ad0>


,participant_id,age,gender,diagnosis,ethnicity,label,FOLD_0,FOLD_1,FOLD_2,FOLD_3,FOLD_4
0,participant_001,62,F,HC,White-British,0,TRAIN,TEST,TRAIN,TRAIN,TRAIN
1,participant_002,58,M,HC,White-British,0,TRAIN,TRAIN,TRAIN,TRAIN,TEST
2,participant_003,69,F,HC,Asian,0,TRAIN,TRAIN,TEST,TRAIN,TRAIN
3,participant_004,64,M,HC,Other,0,TEST,TRAIN,TRAIN,TRAIN,TRAIN
4,participant_005,71,F,HC,White-British,0,TRAIN,TRAIN,TRAIN,TEST,TRAIN
5,participant_006,78,M,Dementia,White-British,2,TRAIN,TRAIN,TRAIN,TEST,TRAIN
6,participant_007,82,F,Dementia,White-British,2,TRAIN,TEST,TRAIN,TRAIN,TRAIN
7,participant_008,75,M,Dementia,Asian,2,TRAIN,TRAIN,TEST,TRAIN,TRAIN
8,participant_009,80,F,Dementia,Mixed,2,TEST,TRAIN,TRAIN,TRAIN,TRAIN
9,participant_010,77,M,Dementia,White-British,2,TRAIN,TRAIN,TRAIN,TRAIN,TEST


In [45]:
stat_result = analyze_metadata(df_metadata_final)
print(stat_result.items())

All data: 10 and diagnosis count: diagnosis
HC          5
Dementia    5
Name: count, dtype: int64
dict_items([('age stat', defaultdict(None, {'HC': {'Mean age': np.float64(64.8), 'Std age': np.float64(4.707440918375928)}, 'Dementia': {'Mean age': np.float64(78.4), 'Std age': np.float64(2.416609194718914)}, 'Overall': {'Mean age': np.float64(71.6), 'Std age': np.float64(7.761443164772902)}})), ('age histograms', defaultdict(None, {'HC': {'0 - 50': np.int64(0), '50 - 60': np.int64(1), '60 - 70': np.int64(3), '70 - 80': np.int64(1), '80 - 100': np.int64(0)}, 'Dementia': {'0 - 50': np.int64(0), '50 - 60': np.int64(0), '60 - 70': np.int64(0), '70 - 80': np.int64(3), '80 - 100': np.int64(2)}})), ('Ethnic diagnosis', defaultdict(None, {'White-British': diagnosis
HC          3
Dementia    3
Name: count, dtype: int64, 'Asian': diagnosis
HC          1
Dementia    1
Name: count, dtype: int64, 'Other': diagnosis
HC    1
Name: count, dtype: int64, 'Mixed': diagnosis
Dementia    1
Name: count, dtype

In [46]:
feat_dict = extract_acoustic_features(df_metadata_final, list_task_name=TASK_CHOSEN, data_path=DATA_PATH, feat_path=FEATS_PATH, list_feature_type=LIST_ACOUSTIC)

Processing feature type: eGeMAPSv02
--------------------------------------------------


Extracting eGeMAPSv02:   0%|          | 0/10 [00:00<?, ?it/s]

audio pattern: /mnt/data_lab513/ducvu/fake-speech-data/audio_files/participant_001/participant_001_Q4.mp3


/tmp/ipykernel_2228950/3726734248.py:90: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_feat = pd.concat([df_feat, df_os_feat], ignore_index=True, sort=False)
Extracting eGeMAPSv02:  10%|█         | 1/10 [00:02<00:18,  2.07s/it]

audio pattern: /mnt/data_lab513/ducvu/fake-speech-data/audio_files/participant_002/participant_002_Q4.mp3


...

Extracting eGeMAPSv02:  90%|█████████ | 9/10 [00:31<00:04,  4.52s/it]

audio pattern: /mnt/data_lab513/ducvu/fake-speech-data/audio_files/participant_010/participant_010_Q4.mp3


Extracting eGeMAPSv02: 100%|██████████| 10/10 [00:32<00:00,  3.28s/it]


Merge features with metadata
Saving feature to: ~/home2/ducvu/speech-processing-implement//feats/cognoSpeak_eGeMAPSv02.csv


OSError: Cannot save file into a non-existent directory: '/home2/ducvu/home2/ducvu/speech-processing-implement/feats'

In [ ]:
feat_dict.items()

In [ ]:
for feat_type, (df, names) in feat_dict.items():
    print(f"df: {df}")

In [ ]:

for feat_type, (df, names) in feat_dict.items():
    print(f"feat_type:{feat_type}")
    print(f"name:{names}")